In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.tseries.offsets import DateOffset
from src import preprocessing, features

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')
FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future



In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = preprocessing.process_store_data(store_df)

# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always
train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])
sales_df = train_df[['Date', 'Store', 'Sales']].copy()
train_df.drop(['Customers', 'Sales'], axis=1)

targets = features.make_targets(sales_df, horizon=FORECAST_HORIZON)

test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
test_df = features.attach_store_data(test_df, store_df)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_4948\1651772828.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])


In [3]:
""" Feature engineering """
lags = [1, 2, 3] # days
windows = [7, 14, 30]


train_df = features.attach_store_data(train_df, store_df)

# Competition-related features
train_df['CompetitionDistance'] = train_df['CompetitionDistance'].apply(np.log1p)
train_df['CompetitionSinceMonths'] = ( (train_df['Date'] - train_df['CompetitionSinceDate']).dt.days / 30.0 ).round()

# Promotion related features
train_df['Promo2SinceWeeks'] = (train_df['Date'] - train_df['Promo2SinceDate']).dt.days / 7.0
train_df['Promo2SinceWeeks'] =  train_df['Promo2SinceWeeks'].fillna(0).round().astype(int) 
train_df['Promo2SinceWeeks'] = train_df['Promo2SinceWeeks'] * train_df['Promo2']

# Basic date features
train_df['WeekOfYear'] = train_df['Date'].dt.isocalendar().week
train_df['Month'] = train_df['Date'].dt.month
train_df['Year'] = train_df['Date'].dt.year
train_df['Quarter'] = train_df['Date'].dt.quarter

# Calendar and seasonality features
train_df['is_weekend'] = train_df['Date'].dt.dayofweek >= 5

# Cyclical features
cyclic_month = features.make_cyclic(train_df['Month'], period=12)
cyclic_week = features.make_cyclic(train_df['DayOfWeek'], period=7)

# Lagged features
date_lags = [DateOffset(days=lag) for lag in lags]
lagged_df = features.make_lags(sales_df, date_lags)

# Rollign window features
window_df = features.make_rolling(sales_df, windows)

# Drop useless
#train_df.drop(['Promo2SinceDate', 'CompetitionSinceDate'], axis=1, inplace=True)


In [112]:
train_df.merge(lagged_df, how='left').head(3)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Month_cos,Dayofweek_sin,Dayofweek_cos,lag_days_1,lag_days_2,lag_days_3,lag_days_4,lag_days_5,lag_days_6,lag_days_7
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,-0.866025,-0.974928,-0.222521,5020.0,4782.0,5011.0,6102.0,0.0,4364.0,3706.0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,-0.866025,-0.974928,-0.222521,5567.0,6402.0,5671.0,6627.0,0.0,2512.0,3854.0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,-0.866025,-0.974928,-0.222521,8977.0,7610.0,8864.0,8107.0,0.0,3878.0,5080.0
